# Data Cleaning

Validate the raw RSNA metadata and construct one patient-level record per chest X-ray for binary pneumonia classification.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "data" / "raw"

TRAIN_IMAGE_DIR = DATA_DIR / "stage_2_train_images"
TRAIN_LABELS_PATH = DATA_DIR / "stage_2_train_labels.csv"
CLASS_INFO_PATH = DATA_DIR / "stage_2_detailed_class_info.csv"

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(TRAIN_LABELS_PATH)
class_df = pd.read_csv(CLASS_INFO_PATH)

print("Training labels shape:", labels_df.shape)
print("Detailed class info shape:", class_df.shape)

Training labels shape: (30227, 6)
Detailed class info shape: (30227, 2)


## 1. Schema and Core Label Validation

Validate the columns required for the classification task and confirm that the binary target contains only valid values.


In [2]:
required_label_columns = {"patientId", "x", "y", "width", "height", "Target"}
required_class_columns = {"patientId", "class"}

missing_label_columns = required_label_columns - set(labels_df.columns)
missing_class_columns = required_class_columns - set(class_df.columns)

if missing_label_columns:
    raise ValueError(f"Missing label columns: {sorted(missing_label_columns)}")

if missing_class_columns:
    raise ValueError(f"Missing class-info columns: {sorted(missing_class_columns)}")

valid_targets = {0, 1}
observed_targets = set(labels_df["Target"].dropna().unique())

if not observed_targets.issubset(valid_targets):
    raise ValueError(f"Unexpected Target values: {observed_targets - valid_targets}")

schema_checks = pd.DataFrame({
    "check": [
        "Required label columns present",
        "Required detailed-class columns present",
        "Missing patientId in labels",
        "Missing Target in labels",
        "Target values restricted to 0/1",
        "Missing patientId in class info",
        "Missing class in class info",
    ],
    "result": [
        len(missing_label_columns) == 0,
        len(missing_class_columns) == 0,
        labels_df["patientId"].isna().sum() == 0,
        labels_df["Target"].isna().sum() == 0,
        observed_targets.issubset(valid_targets),
        class_df["patientId"].isna().sum() == 0,
        class_df["class"].isna().sum() == 0,
    ],
    "count_or_detail": [
        "OK",
        "OK",
        int(labels_df["patientId"].isna().sum()),
        int(labels_df["Target"].isna().sum()),
        sorted(observed_targets),
        int(class_df["patientId"].isna().sum()),
        int(class_df["class"].isna().sum()),
    ],
})

display(schema_checks)

if not schema_checks["result"].all():
    raise ValueError("One or more core schema checks failed.")


,check,result,count_or_detail
0,Required label columns present,True,OK
1,Required detailed-class columns present,True,OK
2,Missing patientId in labels,True,0
3,Missing Target in labels,True,0
4,Target values restricted to 0/1,True,"[0, 1]"
5,Missing patientId in class info,True,0
6,Missing class in class info,True,0


## 2. Exact Duplicate Validation

Check for exact duplicate records. Repeated patient IDs are not automatically treated as duplicates because positive patients may legitimately have multiple bounding-box annotations.


In [3]:
duplicate_label_mask = labels_df.duplicated(keep=False)
duplicate_class_mask = class_df.duplicated(keep=False)

duplicate_summary = pd.DataFrame({
    "table": ["train_labels", "detailed_class_info"],
    "exact_duplicate_rows": [
        int(labels_df.duplicated().sum()),
        int(class_df.duplicated().sum()),
    ],
    "rows_in_duplicate_groups": [
        int(duplicate_label_mask.sum()),
        int(duplicate_class_mask.sum()),
    ],
})

display(duplicate_summary)

,table,exact_duplicate_rows,rows_in_duplicate_groups
0,train_labels,0,0
1,detailed_class_info,3543,6941


## 3. Target and Bounding-Box Consistency

For RSNA detection annotations:

- `Target = 1` should have complete bounding-box coordinates.
- `Target = 0` should not contain bounding-box coordinates.
- Positive bounding boxes must have positive width and height.


In [4]:
bbox_cols = ["x", "y", "width", "height"]

positive_rows = labels_df[labels_df["Target"] == 1].copy()
negative_rows = labels_df[labels_df["Target"] == 0].copy()

positive_missing_bbox = positive_rows[bbox_cols].isna().any(axis=1)
negative_has_bbox = negative_rows[bbox_cols].notna().any(axis=1)
invalid_positive_size = (
    (positive_rows["width"] <= 0) |
    (positive_rows["height"] <= 0)
)

bbox_consistency = pd.DataFrame({
    "check": [
        "Positive rows missing bounding-box values",
        "Negative rows containing bounding-box values",
        "Positive rows with non-positive width/height",
    ],
    "invalid_rows": [
        int(positive_missing_bbox.sum()),
        int(negative_has_bbox.sum()),
        int(invalid_positive_size.sum()),
    ],
})

display(bbox_consistency)

if bbox_consistency["invalid_rows"].sum() != 0:
    raise ValueError("Target/bounding-box consistency problems detected.")

,check,invalid_rows
0,Positive rows missing bounding-box values,0
1,Negative rows containing bounding-box values,0
2,Positive rows with non-positive width/height,0


## 4. Patient-Level Target Consistency

A patient may have multiple bounding-box rows, but all rows belonging to that patient must agree on the binary target.


In [5]:
target_nunique = labels_df.groupby("patientId")["Target"].nunique()
conflicting_target_ids = target_nunique[target_nunique > 1].index

print("Patients with conflicting Target values:", len(conflicting_target_ids))

if len(conflicting_target_ids) != 0:
    raise ValueError("Conflicting Target values found within one or more patients.")

Patients with conflicting Target values: 0


## 5. Detailed-Class Consistency

Repeated rows for a patient should resolve to one diagnostic class, and that class should agree with the binary target:

- `Lung Opacity` → `Target = 1`
- `Normal` → `Target = 0`
- `No Lung Opacity / Not Normal` → `Target = 0`


In [6]:
class_nunique = class_df.groupby("patientId")["class"].nunique()
conflicting_class_ids = class_nunique[class_nunique > 1].index

print("Patients with conflicting detailed classes:", len(conflicting_class_ids))

patient_targets = (
    labels_df.groupby("patientId", as_index=False)
    .agg(Target=("Target", "first"))
)

patient_classes = (
    class_df.groupby("patientId", as_index=False)
    .agg(detailed_class=("class", "first"))
)

target_class_check = patient_targets.merge(
    patient_classes,
    on="patientId",
    how="outer",
    indicator=True,
    validate="one_to_one",
)

expected_target = {
    "Lung Opacity": 1,
    "Normal": 0,
    "No Lung Opacity / Not Normal": 0,
}

target_class_check["expected_target"] = (
    target_class_check["detailed_class"].map(expected_target)
)

target_class_check["mapping_valid"] = (
    (target_class_check["_merge"] == "both") &
    (target_class_check["Target"] == target_class_check["expected_target"])
)

invalid_target_class = target_class_check[
    ~target_class_check["mapping_valid"]
].copy()

class_consistency_summary = pd.DataFrame({
    "metric": [
        "Patients with conflicting detailed classes",
        "Patients missing from either metadata table",
        "Patients with invalid Target/class mapping",
    ],
    "count": [
        len(conflicting_class_ids),
        int((target_class_check["_merge"] != "both").sum()),
        len(invalid_target_class),
    ],
})

display(class_consistency_summary)

if class_consistency_summary["count"].sum() != 0:
    raise ValueError("Detailed-class consistency problems detected.")


Patients with conflicting detailed classes: 0


,metric,count
0,Patients with conflicting detailed classes,0
1,Patients missing from either metadata table,0
2,Patients with invalid Target/class mapping,0


## 6. Training Image and Label Consistency

Verify that every labelled patient has exactly one corresponding training DICOM file and identify any training images that lack labels.


In [7]:
train_dicom_files = sorted(TRAIN_IMAGE_DIR.glob("*.dcm"))
image_ids = {path.stem for path in train_dicom_files}
label_ids = set(patient_targets["patientId"])

missing_images = sorted(label_ids - image_ids)
unlabelled_images = sorted(image_ids - label_ids)

image_consistency = pd.DataFrame({
    "metric": [
        "Unique labelled patients",
        "Training DICOM files",
        "Labelled patients missing DICOM image",
        "Training DICOM images without label",
    ],
    "count": [
        len(label_ids),
        len(train_dicom_files),
        len(missing_images),
        len(unlabelled_images),
    ],
})

display(image_consistency)

if missing_images or unlabelled_images:
    raise ValueError("Training image/label mismatch detected.")


,metric,count
0,Unique labelled patients,26684
1,Training DICOM files,26684
2,Labelled patients missing DICOM image,0
3,Training DICOM images without label,0


## 7. Validate Bounding Boxes Against DICOM Dimensions

Check every positive bounding box against the dimensions stored in its corresponding DICOM image. This is performed before collapsing detection annotations so that malformed source annotations cannot silently enter the cleaned dataset.


In [8]:
positive_patient_ids = positive_rows["patientId"].unique()

image_dimensions = {}

for i, patient_id in enumerate(positive_patient_ids, start=1):
    image_path = TRAIN_IMAGE_DIR / f"{patient_id}.dcm"
    ds = pydicom.dcmread(image_path, stop_before_pixels=True)
    image_dimensions[patient_id] = (int(ds.Columns), int(ds.Rows))

    if i % 1000 == 0 or i == len(positive_patient_ids):
        print(f"Read DICOM dimensions: {i}/{len(positive_patient_ids)}")

bbox_bounds_df = positive_rows[
    ["patientId", "x", "y", "width", "height"]
].copy()

bbox_bounds_df["image_width"] = bbox_bounds_df["patientId"].map(
    lambda pid: image_dimensions[pid][0]
)
bbox_bounds_df["image_height"] = bbox_bounds_df["patientId"].map(
    lambda pid: image_dimensions[pid][1]
)

bbox_bounds_df["x2"] = bbox_bounds_df["x"] + bbox_bounds_df["width"]
bbox_bounds_df["y2"] = bbox_bounds_df["y"] + bbox_bounds_df["height"]

bbox_bounds_df["valid_bounds"] = (
    (bbox_bounds_df["x"] >= 0) &
    (bbox_bounds_df["y"] >= 0) &
    (bbox_bounds_df["width"] > 0) &
    (bbox_bounds_df["height"] > 0) &
    (bbox_bounds_df["x2"] <= bbox_bounds_df["image_width"]) &
    (bbox_bounds_df["y2"] <= bbox_bounds_df["image_height"])
)

invalid_bounds_df = bbox_bounds_df[
    ~bbox_bounds_df["valid_bounds"]
].copy()

print("\nPositive bounding boxes checked:", len(bbox_bounds_df))
print("Bounding boxes outside image bounds:", len(invalid_bounds_df))

if len(invalid_bounds_df) != 0:
    raise ValueError("One or more bounding boxes fall outside DICOM image dimensions.")


Read DICOM dimensions: 1000/6012
Read DICOM dimensions: 2000/6012
Read DICOM dimensions: 3000/6012
Read DICOM dimensions: 4000/6012
Read DICOM dimensions: 5000/6012
Read DICOM dimensions: 6000/6012
Read DICOM dimensions: 6012/6012

Positive bounding boxes checked: 9555
Bounding boxes outside image bounds: 0


## 8. Construct Patient-Level Classification Metadata

Collapse detection-level annotations to one record per chest X-ray.

The resulting metadata contains the patient ID, binary target, detailed diagnostic class, number of pneumonia bounding boxes, and DICOM filename.


In [9]:
box_counts = (
    positive_rows.groupby("patientId")
    .size()
    .rename("num_pneumonia_boxes")
)

clean_df = patient_targets.merge(
    patient_classes,
    on="patientId",
    how="inner",
    validate="one_to_one",
)

clean_df["num_pneumonia_boxes"] = (
    clean_df["patientId"]
    .map(box_counts)
    .fillna(0)
    .astype(int)
)

clean_df["image_filename"] = clean_df["patientId"] + ".dcm"

clean_df = clean_df[
    [
        "patientId",
        "Target",
        "detailed_class",
        "num_pneumonia_boxes",
        "image_filename",
    ]
].sort_values("patientId").reset_index(drop=True)

display(clean_df.head())
print("\nClean patient-level dataset shape:", clean_df.shape)
print("Unique patients:", clean_df["patientId"].nunique())


,patientId,Target,detailed_class,num_pneumonia_boxes,image_filename
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,0,No Lung Opacity / Not Normal,0,0004cfab-14fd-4e49-80ba-63a80b6bddd6.dcm
1,000924cf-0f8d-42bd-9158-1af53881a557,0,Normal,0,000924cf-0f8d-42bd-9158-1af53881a557.dcm
2,000db696-cf54-4385-b10b-6b16fbb3f985,1,Lung Opacity,2,000db696-cf54-4385-b10b-6b16fbb3f985.dcm
3,000fe35a-2649-43d4-b027-e67796d412e0,1,Lung Opacity,2,000fe35a-2649-43d4-b027-e67796d412e0.dcm
4,001031d9-f904-4a23-b3e5-2c088acd19c6,1,Lung Opacity,2,001031d9-f904-4a23-b3e5-2c088acd19c6.dcm



Clean patient-level dataset shape: (26684, 5)
Unique patients: 26684


## 9. Final Validation of Cleaned Metadata

Run strict assertions on the canonical classification dataset before saving it.


In [10]:
final_checks = pd.DataFrame({
    "check": [
        "Exactly one row per patient",
        "No missing patient IDs",
        "No missing targets",
        "No missing detailed classes",
        "All targets are binary",
        "Positive patients have at least one pneumonia box",
        "Negative patients have zero pneumonia boxes",
        "All image files exist",
    ],
    "passed": [
        len(clean_df) == clean_df["patientId"].nunique(),
        clean_df["patientId"].notna().all(),
        clean_df["Target"].notna().all(),
        clean_df["detailed_class"].notna().all(),
        set(clean_df["Target"].unique()).issubset({0, 1}),
        (clean_df.loc[clean_df["Target"] == 1, "num_pneumonia_boxes"] >= 1).all(),
        (clean_df.loc[clean_df["Target"] == 0, "num_pneumonia_boxes"] == 0).all(),
        clean_df["image_filename"].map(
            lambda name: (TRAIN_IMAGE_DIR / name).is_file()
        ).all(),
    ],
})

display(final_checks)

if not final_checks["passed"].all():
    failed = final_checks.loc[~final_checks["passed"], "check"].tolist()
    raise ValueError(f"Final cleaned-metadata validation failed: {failed}")

,check,passed
0,Exactly one row per patient,True
1,No missing patient IDs,True
2,No missing targets,True
3,No missing detailed classes,True
4,All targets are binary,True
5,Positive patients have at least one pneumonia box,True
6,Negative patients have zero pneumonia boxes,True
7,All image files exist,True


## 10. Save Cleaned Metadata

Save the patient-level classification metadata for downstream analysis and modelling.

In [11]:
CLEAN_METADATA_PATH = PROCESSED_DATA_DIR / "clean_classification_metadata.csv"
clean_df.to_csv(CLEAN_METADATA_PATH, index=False)

cleaning_summary = pd.DataFrame({
    "metric": [
        "Raw annotation rows",
        "Clean patient-level rows",
        "Unique patients",
        "Pneumonia patients",
        "Non-pneumonia patients",
        "Positive bounding-box annotations",
        "Patients with multiple pneumonia boxes",
        "Invalid target/bbox rows",
        "Conflicting patient targets",
        "Conflicting detailed classes",
        "Missing labelled images",
        "Unlabelled training images",
        "Bounding boxes outside image bounds",
    ],
    "value": [
        len(labels_df),
        len(clean_df),
        clean_df["patientId"].nunique(),
        int((clean_df["Target"] == 1).sum()),
        int((clean_df["Target"] == 0).sum()),
        len(positive_rows),
        int((clean_df["num_pneumonia_boxes"] > 1).sum()),
        int(bbox_consistency["invalid_rows"].sum()),
        len(conflicting_target_ids),
        len(conflicting_class_ids),
        len(missing_images),
        len(unlabelled_images),
        len(invalid_bounds_df),
    ],
})

display(cleaning_summary)
print(f"\nClean metadata saved to: {CLEAN_METADATA_PATH}")
print("\nDATA CLEANING COMPLETE")


,metric,value
0,Raw annotation rows,30227
1,Clean patient-level rows,26684
2,Unique patients,26684
3,Pneumonia patients,6012
4,Non-pneumonia patients,20672
5,Positive bounding-box annotations,9555
6,Patients with multiple pneumonia boxes,3398
7,Invalid target/bbox rows,0
8,Conflicting patient targets,0
9,Conflicting detailed classes,0



Clean metadata saved to: E:\Projects\CNN and Vision Transformer-Based Pneumonia Classification in Chest X-Rays\outputs\cleaning\clean_classification_metadata.csv

DATA CLEANING COMPLETE
